In [5]:
import os
import time
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np




class PrunableLinear(nn.Module):


    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        bound = 1 / math.sqrt(in_features)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates         = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        with torch.no_grad():
            return torch.sigmoid(self.gate_scores)

    def extra_repr(self):
        return f"in={self.in_features}, out={self.out_features}"



class SelfPruningNet(nn.Module):


    def __init__(self, dropout: float = 0.3):
        super().__init__()
        self.flatten = nn.Flatten()

        self.fc1 = PrunableLinear(3072, 1024)
        self.fc2 = PrunableLinear(1024, 512)
        self.fc3 = PrunableLinear(512,  256)
        self.fc4 = PrunableLinear(256,  10)

        self.bn1 = nn.BatchNorm1d(1024)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(256)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.flatten(x)
        x = self.drop(F.relu(self.bn1(self.fc1(x))))
        x = self.drop(F.relu(self.bn2(self.fc2(x))))
        x = F.relu(self.bn3(self.fc3(x)))
        return self.fc4(x)

    def prunable_layers(self):
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m


    def sparsity_loss(self) -> torch.Tensor:

        all_gates = []
        for layer in self.prunable_layers():
            all_gates.append(torch.sigmoid(layer.gate_scores).reshape(-1))
        all_gates = torch.cat(all_gates)
        return all_gates.mean()   # ← MEAN, not SUM

    def sparsity_level(self, threshold: float = 1e-2) -> float:

        total, pruned = 0, 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            total  += g.numel()
            pruned += (g < threshold).sum().item()
        return 100.0 * pruned / total if total > 0 else 0.0

    def all_gate_values(self) -> np.ndarray:
        return np.concatenate([
            layer.get_gates().cpu().numpy().ravel()
            for layer in self.prunable_layers()
        ])




def build_dataloaders(batch_size: int = 256, num_workers: int = 2):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    test_set  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader




def train_one_epoch(model, loader, optimizer, device, lam):
    model.train()
    total_l = cls_l = spar_l = 0.0
    n = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits      = model(images)
        cls_loss    = F.cross_entropy(logits, labels)
        sparse_loss = model.sparsity_loss()
        loss        = cls_loss + lam * sparse_loss

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_l += loss.item()
        cls_l   += cls_loss.item()
        spar_l  += sparse_loss.item()
        n       += 1

    return total_l / n, cls_l / n, spar_l / n




@torch.no_grad()
def evaluate(model, loader, device) -> float:
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds    = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total




def run_experiment(lam, train_loader, test_loader, device, epochs=50, lr=1e-3):
    print(f"\n{'='*62}")
    print(f"  λ = {lam}   |   epochs = {epochs}")
    print(f"{'='*62}")

    model     = SelfPruningNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tot, cls, spr = train_one_epoch(model, train_loader, optimizer, device, lam)
        acc  = evaluate(model, test_loader, device)
        spar = model.sparsity_level()
        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Ep {epoch:3d}/{epochs}  Loss={tot:.4f}  "
                  f"CLS={cls:.4f}  SPR={spr:.4f}  "
                  f"Acc={acc:.2f}%  Sparse={spar:.1f}%  "
                  f"({time.time()-t0:.1f}s)")

    final_acc  = evaluate(model, test_loader, device)
    final_spar = model.sparsity_level()
    gate_vals  = model.all_gate_values()

    print(f"\n  ✓ FINAL  Accuracy={final_acc:.2f}%   Sparsity={final_spar:.1f}%")
    return {"lam": lam, "accuracy": final_acc, "sparsity": final_spar,
            "gate_values": gate_vals}




def plot_gate_distribution(gate_values, lam, save_path="gate_distribution.png"):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(gate_values, bins=120, color="#2563EB", edgecolor="none", alpha=0.85)
    ax.axvline(0.01, color="red", linestyle="--", lw=1.5, label="prune threshold (0.01)")
    ax.set_xlabel("Gate Value  sigmoid(gate_score)", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_title(
        f"Gate Value Distribution  —  λ = {lam}\n"
        "Spike near 0 = pruned weights  |  Cluster near 1 = retained weights",
        fontsize=11)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  → Saved: {save_path}")


def plot_tradeoff(results, save_path="lambda_tradeoff.png"):
    lambdas    = [str(r["lam"]) for r in results]
    accuracies = [r["accuracy"] for r in results]
    sparsities = [r["sparsity"] for r in results]
    x = np.arange(len(lambdas))
    w = 0.35

    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax2 = ax1.twinx()
    ax1.bar(x - w/2, accuracies, w, label="Test Accuracy (%)", color="#2563EB", alpha=0.85)
    ax2.bar(x + w/2, sparsities, w, label="Sparsity (%)",      color="#DC2626", alpha=0.75)

    ax1.set_xlabel("Lambda (λ)", fontsize=12)
    ax1.set_ylabel("Test Accuracy (%)", color="#2563EB", fontsize=11)
    ax2.set_ylabel("Sparsity Level (%)", color="#DC2626", fontsize=11)
    ax1.set_xticks(x); ax1.set_xticklabels(lambdas)
    ax1.set_title("Accuracy vs Sparsity Trade-off Across λ Values", fontsize=12)
    ax1.set_ylim(0, 100); ax2.set_ylim(0, 100)

    lines  = [plt.Rectangle((0,0),1,1, color="#2563EB"),
              plt.Rectangle((0,0),1,1, color="#DC2626")]
    labels = ["Test Accuracy (%)", "Sparsity Level (%)"]
    ax1.legend(lines, labels, loc="upper right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  → Saved: {save_path}")




def main():
    device = torch.device(
        "cuda" if torch.cuda.is_available() else
        "mps"  if torch.backends.mps.is_available() else
        "cpu"
    )
    print(f"\nDevice: {device}")

    train_loader, test_loader = build_dataloaders(batch_size=256, num_workers=2)

    lambdas = [0.1, 0.3, 0.5]
    epochs  = 50

    results = []
    for lam in lambdas:
        res = run_experiment(lam, train_loader, test_loader, device, epochs=epochs)
        results.append(res)


    print("\n" + "="*52)
    print(f"{'Lambda':>10}  {'Test Accuracy':>15}  {'Sparsity':>10}")
    print("-"*52)
    for r in results:
        print(f"{r['lam']:>10}  {r['accuracy']:>13.2f}%  {r['sparsity']:>9.1f}%")
    print("="*52)


    best = max(results, key=lambda r: r["accuracy"])
    print(f"\nBest model: λ={best['lam']}  acc={best['accuracy']:.2f}%  sparsity={best['sparsity']:.1f}%")

    plot_gate_distribution(best["gate_values"], lam=best["lam"])
    plot_tradeoff(results)

    print("\nDone! Outputs saved.")


if __name__ == "__main__":
    main()


Device: cuda


100%|██████████| 170M/170M [00:04<00:00, 39.6MB/s]



  λ = 0.1   |   epochs = 50
  Ep   1/50  Loss=1.8719  CLS=1.8219  SPR=0.5000  Acc=42.11%  Sparse=0.0%  (19.3s)
  Ep  10/50  Loss=1.4987  CLS=1.4487  SPR=0.5000  Acc=52.43%  Sparse=0.0%  (18.2s)
  Ep  20/50  Loss=1.4122  CLS=1.3622  SPR=0.5000  Acc=55.58%  Sparse=0.0%  (19.5s)
  Ep  30/50  Loss=1.3293  CLS=1.2793  SPR=0.5000  Acc=58.31%  Sparse=0.0%  (18.6s)
  Ep  40/50  Loss=1.2475  CLS=1.1975  SPR=0.5000  Acc=60.82%  Sparse=0.0%  (18.6s)
  Ep  50/50  Loss=1.2183  CLS=1.1683  SPR=0.5000  Acc=60.96%  Sparse=0.0%  (18.8s)

  ✓ FINAL  Accuracy=60.96%   Sparsity=0.0%

  λ = 0.3   |   epochs = 50
  Ep   1/50  Loss=1.9828  CLS=1.8328  SPR=0.5000  Acc=41.00%  Sparse=0.0%  (19.7s)
  Ep  10/50  Loss=1.6054  CLS=1.4555  SPR=0.5000  Acc=52.65%  Sparse=0.0%  (19.4s)
  Ep  20/50  Loss=1.5115  CLS=1.3615  SPR=0.5000  Acc=54.55%  Sparse=0.0%  (18.3s)
  Ep  30/50  Loss=1.4287  CLS=1.2787  SPR=0.5000  Acc=57.92%  Sparse=0.0%  (19.1s)
  Ep  40/50  Loss=1.3461  CLS=1.1962  SPR=0.5000  Acc=60.24%  Sparse